# 04 · Final evaluation with Gemini as the ruler

**Decision this notebook supports:** did v2 fix refusals without breaking accuracy, and is it at your bar (≥80% accuracy, ≤10% hallucination, ≥80% correct refusals, ≤15% over-refusal)?

| | |
|---|---|
| Models | base · SFT v1 · SFT+DPO v1 (main-lab adapters from notebooks 05 and 06) · SFT v2 · SFT v2 + DPO v2 |
| Conditions | `engineered` (no documents) · `rag_rerank` (3 retrieved chunks) · `oracle` (only the right chunk) · `rag_no_gold` (right chunk removed, a retrieval miss) |
| Eval sets | dev_v1 (50, now dev-informed: a regression check) · **dev_v2 (140, the cleaner read)** |
| Scoring | Gemini judge with a fixed rubric (primary), plus the v1 keyword scorer for continuity |

Generation matches the main lab's master evaluation (notebook 13) exactly: greedy, 160 new tokens, float32, the same prompt text. **Truncated answers are scored, not dropped.** The v1 models' dev_v1 answers are reused from the master evaluation (main-lab notebook 13) only when the prompt and settings are byte-identical; otherwise they are generated.

In [ ]:
# Find the v2 folder, put it (and the main lab one folder up) on the import path,
# and apply the memory-safety settings BEFORE torch is imported.
import sys, json, time, re, math, random, hashlib, collections
from pathlib import Path
V2_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/v2.yaml").is_file())
sys.path.insert(0, str(V2_ROOT))
from v2lib import setup, device as D, prompts as P, facts as F
CFG = setup.cfg()
SNAP = setup.read_json(V2_ROOT / CFG["retrieval"]["snapshot"])
CHUNKS = SNAP["chunks"]
print("v2 folder:", V2_ROOT)
print("main lab:", setup.PARENT)
print("device:", D.device(), "| dtype:", D.dtype(), "| base model:", D.base_model_path())
D.memory_report("start")

## Eval sets, conditions and the plan

In [ ]:
import pandas as pd
EVAL = {"dev_v1": setup.read_json(setup.path(CFG["data"]["dev_v1"])), "dev_v2": setup.read_json(V2_ROOT / CFG["data"]["dev_v2"])}
FE = CFG["final_eval"]
BAR = setup.read_json(setup.path(FE["bar_path"]))["bar"] if setup.path(FE["bar_path"]).exists() else FE["bar_default"]
print("bar:", BAR)

def chunks_for(q, condition):
    if condition == "engineered":
        return []
    ranked = SNAP["questions"][q["question"]]["rerank"]
    if condition == "rag_rerank":
        return ranked[:3]
    gold = [c for c in sorted(CHUNKS) if q["answerable"] and q["fact_id"] in CHUNKS[c]["fact_ids"]]
    if condition == "oracle":
        return gold
    if condition == "rag_no_gold":
        return [c for c in ranked if c not in gold][:3]
    raise ValueError(condition)

def cohort(items, condition):
    return [q for q in items if condition in ("engineered", "rag_rerank") or q["answerable"]]

def artifact_ok(spec):
    if spec["kind"] == "base":
        return True, ""
    paths = [setup.path(spec["adapter"])] + ([setup.path(spec["base"])] if spec["kind"] == "merged_adapter" else [])
    missing = [str(p) for p in paths if not p.exists()]
    return not missing, ", ".join(missing)

plan = []
for mid, spec in FE["models"].items():
    ok, missing = artifact_ok(spec)
    for cond in spec["conditions"]:
        for es in FE["eval_sets"]:
            plan.append({"model": mid, "condition": cond, "eval_set": es, "questions": len(cohort(EVAL[es], cond)),
                         "status": "ready" if ok else f"SKIPPED (missing {missing})"})
plan = pd.DataFrame(plan)
display(plan.pivot_table(index="model", columns=["eval_set", "condition"], values="questions", aggfunc="sum"))
print("skipped:", plan[plan.status != "ready"].model.unique().tolist())

## Reuse v1 answers from the master evaluation where nothing differs

A parent answer is reused only when the model, condition, question text, system-prompt hash, chunk IDs and generation settings all match. Everything else is generated fresh.

In [ ]:
PARENT_NAME = {"base": "base", "sft_v1": "sft", "sft_dpo_v1": "sft_dpo"}
GEN = dict(CFG["generation"], device=D.device(), dtype=str(D.dtype()).replace("torch.", ""))
reuse = {}
for mid, pname in PARENT_NAME.items():
    for cond in ["engineered", "rag_rerank", "oracle"]:
        path = setup.path(FE["parent_v1_results"]) / f"{pname}__{cond}.json"
        if not path.exists():
            continue
        saved = json.loads(path.read_text())
        g = saved["generation"]
        if (g["max_new_tokens"], g["do_sample"], g["num_beams"], g["device"], g["dtype"]) != (GEN["max_new_tokens"], False, 1, GEN["device"], GEN["dtype"]):
            continue
        byq = {q["id"]: q for q in EVAL["dev_v1"]}
        for a in saved["answers"]:
            q = byq.get(a["question_id"])
            if not q or a["status"] not in ("ok", "truncated") or a["question"] != q["question"]:
                continue
            ids = chunks_for(q, cond)
            if a["chunk_ids"] != ids or a["system_prompt_sha256"] != P.sha(P.system_prompt([CHUNKS[c] for c in ids])):
                continue
            reuse[(mid, cond, "dev_v1", q["id"])] = {"answer": a["answer"], "truncated": a["status"] == "truncated",
                                                     "completion_tokens": a.get("completion_tokens"), "source": "master eval"}
print(len(reuse), "v1 answers reused from the master evaluation (0 if you have not run main-lab notebook 13)")

## Generation: identical to the master evaluation

One answer per (model, condition, question). Each answer is cached on disk the moment it is produced, so the run can stop and resume.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
CACHE = V2_ROOT / FE["cache_dir"]; CACHE.mkdir(parents=True, exist_ok=True)
tok = AutoTokenizer.from_pretrained(D.base_model_path())

def file_sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def model_identity(spec):
    ident = {"kind": spec["kind"]}
    if spec["kind"] != "base":
        ident["adapter"] = file_sha(setup.path(spec["adapter"]) / "adapter_model.safetensors")
    if spec["kind"] == "merged_adapter":
        ident["base"] = file_sha(setup.path(spec["base"]) / "model.safetensors")
    return ident

def load(spec):
    if spec["kind"] == "merged_adapter":
        base = AutoModelForCausalLM.from_pretrained(str(setup.path(spec["base"])), dtype=D.dtype()).to(D.device())
    else:
        base = AutoModelForCausalLM.from_pretrained(D.base_model_path(), dtype=D.dtype()).to(D.device())
    m = base if spec["kind"] == "base" else PeftModel.from_pretrained(base, str(setup.path(spec["adapter"])))
    return m.to(D.device()).eval()

def generate_one(m, q, ids):
    prompt = tok.apply_chat_template(P.messages(q["question"], [CHUNKS[c] for c in ids]), tokenize=True, add_generation_prompt=True, return_dict=False)
    assert len(prompt) <= GEN["max_input_tokens"]
    x = torch.tensor([prompt], device=D.device())
    eos = m.generation_config.eos_token_id
    eos_ids = [e for e in (eos if isinstance(eos, list) else [eos]) if e is not None]
    with torch.inference_mode():
        out = m.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=GEN["max_new_tokens"], do_sample=False,
                         num_beams=1, eos_token_id=eos, pad_token_id=tok.pad_token_id or tok.eos_token_id, use_cache=True)
    new = out[0, x.shape[1]:].tolist()
    return {"answer": tok.decode(new, skip_special_tokens=True).strip(), "completion_tokens": len(new),
            "truncated": len(new) >= GEN["max_new_tokens"] and (not new or new[-1] not in eos_ids), "prompt_tokens": len(prompt)}

In [ ]:
ANSWERS = {}
t_all = time.time()
for mid, spec in FE["models"].items():
    ok, missing = artifact_ok(spec)
    if not ok:
        print(f"SKIPPED {mid}: missing {missing}"); continue
    ident = model_identity(spec)
    jobs = []
    for cond in spec["conditions"]:
        for es in FE["eval_sets"]:
            for q in cohort(EVAL[es], cond):
                ids = chunks_for(q, cond)
                key = hashlib.sha256(json.dumps({"model": ident, "cond": cond, "q": q["question"], "chunks": ids, "gen": GEN}, sort_keys=True).encode()).hexdigest()
                jobs.append((cond, es, q, ids, key))
    todo = [j for j in jobs if (mid, j[0], j[1], j[2]["id"]) not in reuse and not (CACHE / f"{j[4]}.json").exists()]
    print(f"{mid}: {len(jobs)} answers, {len(jobs) - len(todo)} cached or reused, {len(todo)} to generate")
    m = load(spec) if todo else None
    t0 = time.time()
    for n, (cond, es, q, ids, key) in enumerate(jobs):
        k = (mid, cond, es, q["id"])
        if k in reuse:
            ANSWERS[k] = dict(reuse[k], chunk_ids=ids); continue
        path = CACHE / f"{key}.json"
        if path.exists():
            ANSWERS[k] = json.loads(path.read_text()); continue
        rec = generate_one(m, q, ids)
        rec.update(chunk_ids=ids, source="generated", model=mid, condition=cond, eval_set=es, question_id=q["id"])
        path.write_text(json.dumps(rec, ensure_ascii=False))
        ANSWERS[k] = rec
        if (n + 1) % 50 == 0:
            setup.progress("04_final_eval", f"{mid} {n + 1}/{len(jobs)} | {(time.time() - t0) / 60:.1f} min")
        if (n + 1) % 100 == 0:
            print(f"   {mid} {n + 1}/{len(jobs)} | {(time.time() - t0) / 60:.1f} min")
    if m is not None:
        del m
        D.empty_cache(); D.memory_report(f"{mid} done")
print(f"{len(ANSWERS)} answers ready in {(time.time() - t_all) / 60:.1f} min")

## Scoring 1 · the v1 keyword scorer (continuity)

The same `lab.score_answer` used for every v1 number. It is kept so v1 and v2 can be compared on the old ruler too. Unlike the master evaluation, truncated answers are scored here, not dropped.

In [ ]:
from src import lab
fidx = lab.fact_index()
QIDX = {(es, q["id"]): q for es, items in EVAL.items() for q in items}
rows = []
for (mid, cond, es, qid), a in ANSWERS.items():
    q = QIDX[(es, qid)]
    s = lab.score_answer(a["answer"], q, fidx)
    rows.append({"model": mid, "condition": cond, "eval_set": es, "question_id": qid, "answerable": q["answerable"],
                 "category": q["category"], "style": q["style"], "answer": a["answer"], "truncated": a.get("truncated", False),
                 "source": a.get("source"), "legacy_correct": s["correct"], "legacy_refused": s["refused"], "legacy_hallucinated": s["hallucinated"]})
R = pd.DataFrame(rows)
print(R.groupby(["eval_set", "source"]).size())

## Scoring 2 · Gemini as the ruler

First the calibration gate (60 answers with known labels; at least 90% agreement or stop). Then every distinct (question, answer) pair is judged once; identical answers share a verdict. For `rag_no_gold`, the question is graded as answerable: refusing or giving the correct value are both safe, and only a wrong value is a hallucination.

In [ ]:
from v2lib import judge as J, gemini_io as G
cap = CFG["judge"]["max_api_calls"]["final_eval"]
cal = setup.read_json(V2_ROOT / CFG["data"]["judge_calibration"])
cal_res = J.judge(cal, purpose="judge_calibration", cap=cap)
agree = sum(cal_res[it["id"]]["label"] == it["expected"] for it in cal) / len(cal)
print(f"judge calibration agreement: {agree:.1%}")
assert agree >= CFG["judge"]["calibration_min_agreement"]

uniq = {}
for r in rows:
    q = QIDX[(r["eval_set"], r["question_id"])]
    jid = hashlib.sha256((r["eval_set"] + q["id"] + r["answer"]).encode()).hexdigest()[:20]
    r["jid"] = jid
    if jid not in uniq:
        uniq[jid] = {"id": jid, "question": q["question"], "answerable": q["answerable"],
                     "gold": J.gold_for(F.FIDX[q["fact_id"]]) if q["answerable"] else None,
                     "answer": r["answer"], "truncated": r["truncated"]}
print(f"{len(rows)} answers -> {len(uniq)} distinct (question, answer) pairs to judge")
setup.progress("04_final_eval", f"judging {len(uniq)} distinct answers")
VERDICT = J.judge(list(uniq.values()), purpose="final_eval", cap=cap)
R["label"] = [VERDICT[r["jid"]]["label"] for r in rows]
R["rationale"] = [VERDICT[r["jid"]]["rationale"] for r in rows]
print(G.stats())
print(R["label"].value_counts())

## Scoreboard (Gemini ruler)

Wilson 95% intervals show how much each number could move by chance. With 30–90 questions per cell, a few points' difference is noise.

In [ ]:
def wilson(k, n, z=1.96):
    if not n: return (None, None)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(c - h, 3), round(c + h, 3))

def board(df):
    out = []
    for (es, mid, cond), g in df.groupby(["eval_set", "model", "condition"]):
        m = J.metrics(g[["answerable", "label"]].to_dict("records"))
        ans, una = g[g.answerable], g[~g.answerable]
        k_acc = int(ans.label.isin(["correct", "correct_with_unsupported_caveat"]).sum())
        k_ref = int((una.label == "refused").sum())
        row = {"eval_set": es, "model": mid, "condition": cond, "n_ans": len(ans), "n_unans": len(una),
               "accuracy": m["accuracy_answerable"], "acc_95ci": wilson(k_acc, len(ans)),
               "clean_acc": m["clean_accuracy_answerable"], "caveat": m["caveat_rate_answerable"],
               "hallucination": m["hallucination_rate_answerable"], "over_refusal": m["over_refusal_rate"],
               "approp_refusal": m["appropriate_refusal_rate"], "refusal_95ci": wilson(k_ref, len(una)) if len(una) else None,
               "invented_unans": m["invented_rate_unanswerable"], "truncated": m["truncated_rate"],
               "legacy_acc": ans.legacy_correct.mean() if len(ans) else None,
               "legacy_refusal": una.legacy_correct.mean() if len(una) else None}
        if cond in ("engineered", "rag_rerank") and len(una):
            row["meets_bar"] = (row["accuracy"] >= BAR["accuracy_answerable"] and row["hallucination"] <= BAR["hallucination_rate_answerable"]
                                and row["approp_refusal"] >= BAR["appropriate_refusal_rate"] and row["over_refusal"] <= BAR["max_over_refusal"])
        out.append(row)
    return pd.DataFrame(out)

SB = board(R)
order = list(FE["models"])
SB["model"] = pd.Categorical(SB["model"], order, ordered=True)
pd.set_option("display.width", 250)
for es in FE["eval_sets"]:
    print(f"\n===== {es} =====")
    display(SB[SB.eval_set == es].sort_values(["condition", "model"]).drop(columns="eval_set").round(3))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, es in zip(axes, FE["eval_sets"]):
    s = SB[(SB.eval_set == es) & (SB.condition == "rag_rerank")].set_index("model").reindex(order)
    s[["accuracy", "approp_refusal", "hallucination"]].plot.bar(ax=ax, rot=20, title=f"{es} · rag_rerank")
    ax.axhline(0.8, ls="--", c="grey", lw=1); ax.set_ylim(0, 1.05)
plt.tight_layout()
FIG = V2_ROOT / FE["output_dir"]; FIG.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG / "rag_rerank_summary.png", dpi=130); plt.show()

## Where refusals succeed or fail: by category

The v1 weak spots were money and policy near-misses. Each cell shows the correct-refusal rate (n in brackets).

In [ ]:
una = R[~R.answerable]
cat = una.groupby(["eval_set", "category", "condition", "model"]).apply(
    lambda g: f"{(g.label == 'refused').mean():.0%} ({len(g)})", include_groups=False).unstack("model")
display(cat.reindex(columns=[m for m in order if m in cat.columns]))

## Answer accuracy by phrasing style

v1 broke on typos and on trap words. Rows are style; values are accuracy on answerable questions.

In [ ]:
ans = R[R.answerable & R.condition.isin(["engineered", "rag_rerank"])]
sty = ans.groupby(["eval_set", "condition", "style", "model"]).apply(
    lambda g: f"{g.label.isin(['correct', 'correct_with_unsupported_caveat']).mean():.0%} ({len(g)})", include_groups=False).unstack("model")
display(sty.reindex(columns=[m for m in order if m in sty.columns]))

## Retrieval miss (`rag_no_gold`): safe or hallucinated?

The right chunk was removed. **Refused** and **correct from memory** are both safe; **wrong value** is the failure that matters.

In [ ]:
ng = R[R.condition == "rag_no_gold"]
tab = ng.groupby(["eval_set", "model"]).label.value_counts(normalize=True).unstack(fill_value=0).round(2)
display(tab.reindex(columns=[c for c in ["refused", "correct", "correct_with_unsupported_caveat", "wrong_value", "off_topic", "truncated_incomplete"] if c in tab.columns]))

## Paired comparison: SFT v1 → SFT v2 on the same questions

Fixed = wrong in v1 and right in v2; broke = the reverse. "Right" means `correct` for answerable questions and `refused` for unanswerable ones.

In [ ]:
def right(label, answerable):
    return label in ("correct", "correct_with_unsupported_caveat") if answerable else label == "refused"
R["right"] = [right(l, a) for l, a in zip(R.label, R.answerable)]
pairs_out = []
for a, b in [("sft_v1", "sft_v2"), ("sft_dpo_v1", "sft_dpo_v2"), ("sft_v2", "sft_dpo_v2")]:
    for (es, cond), g in R.groupby(["eval_set", "condition"]):
        x = g[g.model == a].set_index("question_id"); y = g[g.model == b].set_index("question_id")
        common = x.index.intersection(y.index)
        if not len(common):
            continue
        fixed = [qid for qid in common if not x.loc[qid, "right"] and y.loc[qid, "right"]]
        broke = [qid for qid in common if x.loc[qid, "right"] and not y.loc[qid, "right"]]
        same = int((x.loc[common, "answer"] == y.loc[common, "answer"]).sum())
        pairs_out.append({"from": a, "to": b, "eval_set": es, "condition": cond, "n": len(common), "fixed": len(fixed),
                          "broke": len(broke), "net": len(fixed) - len(broke), "identical_answers": same,
                          "broke_ids": broke[:8]})
PAIRS = pd.DataFrame(pairs_out)
with pd.option_context("display.max_colwidth", 80):
    display(PAIRS)

## Judge vs the v1 keyword scorer

Where the two rulers disagree, read the answer. The keyword scorer cannot see wrong-model values that happen to contain the right number, or refusals worded differently.

In [ ]:
R["judge_right"] = R["right"]
R["legacy_right"] = R["legacy_correct"]
agree_rate = (R.judge_right == R.legacy_right).mean()
print(f"judge and keyword scorer agree on {agree_rate:.1%} of {len(R)} answers")
dis = R[R.judge_right != R.legacy_right]
with pd.option_context("display.max_colwidth", None):
    display(dis.sample(min(20, len(dis)), random_state=0)[["eval_set", "model", "condition", "question_id", "answer", "label", "legacy_right", "rationale"]])

## Read the answers yourself

Pick any question ID to see every model and condition side by side, with the judge's label.

In [ ]:
def inspect(qid, eval_set="dev_v2"):
    q = QIDX[(eval_set, qid)]
    print(f"{qid} · {q['question']}\nanswerable={q['answerable']} · category={q['category']} · expected={q.get('expected_value')}")
    g = R[(R.eval_set == eval_set) & (R.question_id == qid)][["model", "condition", "label", "answer"]]
    with pd.option_context("display.max_colwidth", None):
        display(g.sort_values(["condition", "model"]))
inspect("dv2_u02"); inspect("dv2_a43")

## Save everything

In [ ]:
OUT = V2_ROOT / FE["output_dir"]
R.drop(columns=["jid"]).to_json(OUT / "answers_scored.json", orient="records", indent=1, force_ascii=False)
SB.assign(model=SB.model.astype(str)).to_csv(OUT / "scoreboard.csv", index=False)
PAIRS.to_csv(OUT / "paired.csv", index=False)
setup.write_json(OUT / "summary.json", {"scoreboard": json.loads(SB.assign(model=SB.model.astype(str)).to_json(orient="records")),
    "paired": json.loads(PAIRS.to_json(orient="records")), "judge_agreement_with_legacy": agree_rate,
    "judge_calibration": agree, "gemini": G.stats(), "generation": GEN, "answers": len(R),
    "reused_from_master_eval": int((R.source == "master eval").sum()), "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")})
print("saved to", OUT.relative_to(V2_ROOT))

## How to read these results

- **dev_v2 is the main result.** dev_v1 informed the v2 design, so v2's dev_v1 score is a regression check, not proof.
- **Small samples.** 60 answerable and 80 unanswerable questions in dev_v2 give 95% intervals of roughly ±10 points. Differences inside the interval are not evidence.
- **The judge is a model too.** It passed a 60-item calibration, but read the disagreements above before trusting a borderline number.
- **One more test remains.** `evals/heldout.json` is untouched; run it once, after all choices are fixed.